## Installs

In [1]:
!pip install datasets
!pip install transformers

## Imports

In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, EarlyStoppingCallback, TrainerCallback
from sklearn.model_selection import train_test_split
from datasets import Dataset
import kagglehub

## Data downloading

In [3]:
path = kagglehub.dataset_download("henryshan/car-review-analysis")
path

100%|██████████| 8.52M/8.52M [00:00<00:00, 59.9MB/s]

Extracting files...


'/root/.cache/kagglehub/datasets/henryshan/car-review-analysis/versions/1'

In [4]:
import os

print(os.listdir(path))

['carreview.csv']


In [5]:
# DATA
df = pd.read_csv(os.path.join(path, "carreview.csv"))
reviews = df["Review"].tolist()

## Data preporating

In [6]:
dataset = Dataset.from_dict({"text": reviews})
dataset

Dataset({
    features: ['text'],
    num_rows: 36984
})

In [7]:
#MODEL NAME
model_name = "openai-community/gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [8]:
def tokenize(example):
    tokens = tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/36984 [00:00<?, ? examples/s]

In [9]:
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.2)

train_dataset = tokenized_dataset["train"]
eval_dataset = tokenized_dataset["test"]

train_dataset

Dataset({
    features: ['text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 29587
})

## Training process

In [10]:
model = AutoModelForCausalLM.from_pretrained(model_name)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
from huggingface_hub import login

login(token='PUT_YOUR_TOKEN')

In [12]:
repository_name = 'gpt-car-recommender'

class PushToHubEvery5EpochsCallback(TrainerCallback):
    def on_epoch_end(self, args, state, control, **kwargs):
        if state.epoch and int(state.epoch) % 5 == 0:
            kwargs["model"].push_to_hub(repository_name, commit_message=f"Checkpoint at epoch {int(state.epoch)} with cars reviews dataset")
        return control

In [13]:
early_stopping = EarlyStoppingCallback(early_stopping_patience=2)

In [14]:
training_args = TrainingArguments(
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    num_train_epochs=20,
    lr_scheduler_type="cosine",
    report_to="none",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    callbacks=[early_stopping, PushToHubEvery5EpochsCallback()],
)

<ipython-input-15-05222ba6fe91>:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [16]:
trainer.train()

`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 